# Nexus: Training an LLM for Compute Cluster Negotiation

This notebook demonstrates training a small LLM to play the Nexus negotiation environment
using **Unsloth** (2x faster LoRA fine-tuning) and **TRL GRPOTrainer** (Group Relative Policy Optimization).

The model learns to:
- Allocate compute resources to jobs based on reward/deadline urgency
- Trade resources on the market (bid/offer)
- Form coalitions for collaborative jobs
- Maximize cumulative score over 10-round episodes

In [ ]:
# Cell 1: Install dependencies
!pip install -q openenv>=0.2.1 unsloth trl transformers datasets peft accelerate bitsandbytes

In [ ]:
# Cell 2: Install Nexus (run from nexus/notebooks or set NEXUS_ROOT for Colab)
from pathlib import Path
import os
NEXUS_ROOT = Path.cwd().parent if (Path.cwd() / "nexus_training.ipynb").exists() else Path.cwd()
os.chdir(NEXUS_ROOT)
!pip install -e .

In [ ]:
# Cell 3: Load model with Unsloth (4-bit quantized for Colab)
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
    max_seq_length=2048,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
)
print(f"Model loaded. Trainable params: {model.print_trainable_parameters()}")

In [ ]:
# Cell 4: Start Nexus OpenEnv server in background
import subprocess, time, sys
from pathlib import Path
NEXUS_ROOT = Path.cwd().parent if (Path.cwd() / "nexus_training.ipynb").exists() else Path.cwd()
sys.path.insert(0, str(NEXUS_ROOT))

server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "nexus_openenv.server.app:app",
     "--host", "0.0.0.0", "--port", "8000"],
    cwd=str(NEXUS_ROOT),
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(5)
print(f"Server started (PID: {server.pid})")

In [ ]:
# Cell 5: Verify server is running
from nexus_openenv.client import NexusEnv
from nexus_openenv.models import NexusAction

# Quick smoke test
with NexusEnv(base_url="http://localhost:8000") as env:
    result = env.reset()
    print("Reset OK. Observation preview:")
    print(result.observation.observation_text[:300])
    print(f"\nRound: {result.observation.round_number}/{result.observation.total_rounds}")
    
    # Take one step
    result = env.step(NexusAction(raw_text="pass"))
    print(f"\nAfter step - Round: {result.observation.round_number}, Reward: {result.reward:.2f}")

In [ ]:
# Cell 6: Build training dataset from environment observations
from datasets import Dataset

SYSTEM_PROMPT = """You are an AI agent playing the Nexus compute cluster negotiation game.
You control a team competing for compute resources (GPU, CPU, memory, bandwidth).
Your goal is to complete jobs before their deadlines by allocating resources.
You can also trade resources on the market and form coalitions.

Respond with JSON actions like:
- {"type": "allocate", "job_id": "J-001"}
- {"type": "bid", "resource_type": "gpu", "quantity": 5, "price": 100}
- {"type": "offer", "resource_type": "cpu", "quantity": 10, "price": 30}
- {"type": "pass"}
You can return multiple actions as a JSON array."""

prompts = []
for i in range(50):
    with NexusEnv(base_url="http://localhost:8000") as env:
        result = env.reset()
        obs_text = result.observation.observation_text
        prompt = f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{obs_text}<|im_end|>\n<|im_start|>assistant\n"
        prompts.append({"prompt": prompt})

dataset = Dataset.from_list(prompts)
print(f"Created dataset with {len(dataset)} prompts")
print(f"Sample prompt length: {len(prompts[0]['prompt'])} chars")

In [ ]:
# Cell 7: Define reward function
import torch

def nexus_reward_fn(prompts, completions, **kwargs):
    """Score model completions by running them through the Nexus environment."""
    rewards = []
    for prompt, completion in zip(prompts, completions):
        try:
            # Extract just the generated text
            if isinstance(completion, list):
                text = tokenizer.decode(completion, skip_special_tokens=True)
            else:
                text = completion
            
            with NexusEnv(base_url="http://localhost:8000") as env:
                env.reset()
                result = env.step(NexusAction(raw_text=text))
                reward = result.reward or 0.0
        except Exception:
            reward = -1.0  # Penalty for unparseable output
        
        rewards.append(reward)
    
    return rewards

print("Reward function defined.")

In [ ]:
# Cell 8: Configure and run GRPOTrainer
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir="./nexus_grpo_output",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    num_generations=4,
    max_completion_length=256,
    max_prompt_length=1024,
    logging_steps=1,
    save_steps=50,
    bf16=True,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    reward_funcs=[nexus_reward_fn],
    processing_class=tokenizer,
)

print("Starting GRPO training...")
trainer.train()
print("Training complete!")

In [ ]:
# Cell 9: Save the trained LoRA adapter
model.save_pretrained("nexus_grpo_lora")
tokenizer.save_pretrained("nexus_grpo_lora")
print("Model saved to nexus_grpo_lora/")

In [ ]:
# Cell 10: Test the trained model
FastLanguageModel.for_inference(model)

with NexusEnv(base_url="http://localhost:8000") as env:
    result = env.reset()
    obs_text = result.observation.observation_text
    
    prompt = f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{obs_text}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.7, do_sample=True)
    
    action_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print("Model output:")
    print(action_text)
    print()
    
    # Execute the action
    result = env.step(NexusAction(raw_text=action_text))
    print(f"Round reward: {result.reward:.2f}")
    print(f"Cumulative score: {result.observation.score:.2f}")
    print(f"\nNext observation preview:")
    print(result.observation.observation_text[:200])

In [ ]:
# Cell 11: Run a full episode with the trained model
print("=== Full Episode with Trained Model ===")

with NexusEnv(base_url="http://localhost:8000") as env:
    result = env.reset()
    total_reward = 0.0
    
    for round_num in range(10):
        obs_text = result.observation.observation_text
        prompt = f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{obs_text}<|im_end|>\n<|im_start|>assistant\n"
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to("cuda")
        
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.7, do_sample=True)
        
        action_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        result = env.step(NexusAction(raw_text=action_text))
        
        round_reward = result.reward or 0.0
        total_reward += round_reward
        print(f"Round {round_num + 1}: reward={round_reward:.2f}, cumulative={result.observation.score:.2f}")
        
        if result.done:
            break
    
    print(f"\nFinal score: {result.observation.score:.2f}")
    print(f"Total reward: {total_reward:.2f}")

In [ ]:
# Cell 12: Cleanup
server.terminate()
print("Server stopped.")